# Загрузим и обработаем данные

In [13]:
import pandas as pd

df = pd.read_csv('stroke_prediction_dataset.csv')
# Выведем голову
print(df.head())
print(df.info())
print(df.describe(include='all').T)

   Patient ID       Patient Name  Age Gender  Hypertension  Heart Disease  \
0       18153    Mamooty Khurana   56   Male             0              1   
1       62749  Kaira Subramaniam   80   Male             0              0   
2       32145      Dhanush Balan   26   Male             1              1   
3        6154        Ivana Baral   73   Male             0              0   
4       48973  Darshit Jayaraman   51   Male             1              1   

  Marital Status      Work Type Residence Type  Average Glucose Level  ...  \
0        Married  Self-employed          Rural                 130.91  ...   
1         Single  Self-employed          Urban                 183.73  ...   
2        Married   Never Worked          Rural                 189.00  ...   
3        Married   Never Worked          Urban                 185.29  ...   
4       Divorced  Self-employed          Urban                 177.34  ...   

     Alcohol Intake Physical Activity Stroke History Family History 

# Обработаем категориальные данные

In [14]:
object_column = df.select_dtypes(include='object').columns
for column in object_column:
    print(column)
    print(df[column].unique())

Patient Name
['Mamooty Khurana' 'Kaira Subramaniam' 'Dhanush Balan' ... 'Ivana Kaur'
 'Anvi Mannan' 'Gokul Trivedi']
Gender
['Male' 'Female']
Marital Status
['Married' 'Single' 'Divorced']
Work Type
['Self-employed' 'Never Worked' 'Private' 'Government Job']
Residence Type
['Rural' 'Urban']
Smoking Status
['Non-smoker' 'Formerly Smoked' 'Currently Smokes']
Alcohol Intake
['Social Drinker' 'Never' 'Rarely' 'Frequent Drinker']
Physical Activity
['Moderate' 'Low' 'High']
Family History of Stroke
['Yes' 'No']
Dietary Habits
['Vegan' 'Paleo' 'Pescatarian' 'Gluten-Free' 'Vegetarian' 'Non-Vegetarian'
 'Keto']
Blood Pressure Levels
['140/108' '146/91' '154/97' ... '157/103' '112/104' '100/69']
Cholesterol Levels
['HDL: 68, LDL: 133' 'HDL: 63, LDL: 70' 'HDL: 59, LDL: 95' ...
 'HDL: 31, LDL: 125' 'HDL: 74, LDL: 123' 'HDL: 35, LDL: 183']
Symptoms
['Difficulty Speaking, Headache'
 'Loss of Balance, Headache, Dizziness, Confusion' 'Seizures, Dizziness'
 ...
 'Numbness, Blurred Vision, Severe Fatigu

Дропаем Patient name
Разбиваем на категории по диапозонам Blood Pressure
Тоже самое с холестерином


In [44]:
import pandas as pd
import json
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer

# Исходный датафрейм
df_ml = df.drop(columns=['Patient Name', 'Patient ID'], axis=1)

# Разбиваем давление
df_ml[['systolic', 'diastolic']] = df_ml['Blood Pressure Levels'].str.split('/', expand=True).astype(int)
df_ml = df_ml.drop('Blood Pressure Levels', axis=1)

# Инициализируем словарь логирования
label_mappings = {}

# Разбиваем симптомы
df_ml['Symptoms'] = df_ml['Symptoms'].fillna('')
df_ml['Symptoms_list'] = df_ml['Symptoms'].str.split(', ')

# Мультиэнкодинг симптомов
mlb = MultiLabelBinarizer()
symptoms_encoded = mlb.fit_transform(df_ml['Symptoms_list'])
symptoms_df = pd.DataFrame(symptoms_encoded, columns=mlb.classes_)
df_ml = df_ml.join(symptoms_df)

# Сохраняем список всех возможных симптомов
label_mappings['Symptoms'] = {symptom: 1 for symptom in mlb.classes_}

# Обрабатываем холестерин
df_ml[['HDL', 'LDL']] = df_ml['Cholesterol Levels'].str.split(', ', expand=True)
df_ml = df_ml.drop('Cholesterol Levels', axis=1)
df_ml['HDL'] = df_ml['HDL'].str.extract(r'(\d+)').astype(int)
df_ml['LDL'] = df_ml['LDL'].str.extract(r'(\d+)').astype(int)

# Удаляем пустые колонки (если вдруг остались)
df_ml = df_ml.drop(columns='', errors='ignore')

# Кодируем категориальные признаки
encoder = LabelEncoder()
categ_columns = df_ml.select_dtypes(include='object').columns

# Симптомы и уже обработанные признаки не кодируем повторно
skip_columns = set([
    'Symptoms', 'Symptoms_list', 'Cholesterol Levels',
    'Blurred Vision', 'Confusion', 'Difficulty Speaking', 'Dizziness',
    'Headache', 'Loss of Balance', 'Numbness', 'Seizures',
    'Severe Fatigue', 'Weakness'
])

for column in categ_columns:
    if column not in skip_columns:
        encoder.fit(df_ml[column])
        df_ml[column] = encoder.transform(df_ml[column])

        # Сохраняем соответствие: оригинал → код
        mapping = {k: int(v) for k, v in zip(encoder.classes_, encoder.transform(encoder.classes_))}
        label_mappings[column] = mapping

# Удаляем вспомогательные признаки
df_ml = df_ml.drop(columns=['Symptoms', 'Symptoms_list'], axis=1)

# Сохраняем подготовленный DataFrame
df_ml.to_csv('df_ml.csv', index=False)

# Сохраняем отображения в JSON
with open('label_mappings.json', 'w', encoding='utf-8') as f:
    json.dump(label_mappings, f, indent=4, ensure_ascii=False)

print("Готово: DataFrame и словарь кодировок сохранены.")

Готово: DataFrame и словарь кодировок сохранены.
